### LMDFit Baseline

This notebook reproduces the LMDFit baseline represented by `AgusDatasetBasedCompare/lmdfitAgus.py`, with standardized datasets

<b> Baseline architecture</b>

The paper specifies **adjusted Fisher-Pearson skewness**, so the paper-consistent SciPy setting is:

```python
skew(cosine_scores, bias=False)
```

`lmdfitAgus.py` is the closest local implementation because it returns skewness, but its current `bias=True` is corrected here. This baseline does not use Patil Eta, labels, Same/Diff pairs, or histogram peak height.


In [14]:
import numpy as np
import pandas as pd
import time
from codecarbon import EmissionsTracker, OfflineEmissionsTracker
import re
from transformers import BertModel, BertTokenizer
import torch
from scipy.stats import skew
from scipy import stats
from sklearn.cluster import KMeans
from sklearn.cluster import AgglomerativeClustering
from tabulate import tabulate
import matplotlib.pyplot as plt
import torch.nn.functional as F
from collections import Counter, defaultdict
from itertools import combinations

### 1.0 LMDFit Architecture

In [15]:
BATCH_SIZE = 64
MAX_TOKEN = 512
RANDOM_SEED = 42
NUM_REPETITIONS = 3
NUM_QUANTILES = 100
SAMPLE_SIZE = 200
TEXT_COL = "text"
# 明确使用第一块 CUDA GPU；不可用时停止，不静默退回 CPU。
DEVICE = "cuda:0"   

In [16]:
def sample_from_length_groups(
    df,
    text_col,
    sample_size=SAMPLE_SIZE,
    num_quantiles=NUM_QUANTILES,
    random_seed=RANDOM_SEED,
    length_fn=None,
    length_col="length",
    group_col="quantiles",
    return_grouped=False,
):
    if len({text_col, length_col, group_col}) != 3:
        raise ValueError("Text, length and group column names must be different")
    if length_col in df.columns or group_col in df.columns:
        raise ValueError("Length/group output columns already exist; choose unused output names")
    if text_col not in df.columns:
        raise ValueError(f"Missing text column: {text_col}")
    for name, value in [("sample_size", sample_size), ("num_quantiles", num_quantiles)]:
        if isinstance(value, bool) or not isinstance(value, (int, np.integer)) or value < 1:
            raise ValueError(f"{name} must be a positive integer")

    text = df[text_col].fillna("").astype(str).str.strip()
    pool = df.loc[text.ne("")].copy().reset_index(drop=True)
    text = text.loc[text.ne("")].reset_index(drop=True)
    if sample_size > len(pool):
        raise ValueError(f"Requested {sample_size} rows, but only {len(pool)} valid texts")

    lengths = text.str.split().str.len() if length_fn is None else text.map(length_fn)
    lengths = pd.to_numeric(lengths, errors="raise")
    if not np.isfinite(lengths.to_numpy(dtype=float)).all() or lengths.lt(0).any():
        raise ValueError("Text lengths must be finite, non-negative numbers")

    q = min(num_quantiles, sample_size, lengths.nunique())
    # 相同长度不拆组；所有文本长度相同时使用一个组。
    groups = (
        pd.Series(np.zeros(len(pool), dtype=int))
        if q <= 1 else pd.qcut(lengths, q=q, labels=False, duplicates="drop")
    )
    # 分组基于完整有效数据，而不是抽样后的数据。
    pool[length_col] = lengths.to_numpy()
    pool[group_col] = pd.Categorical(
        groups.map(lambda group: f"Q{int(group) + 1}"),
        categories=[f"Q{int(group) + 1}" for group in sorted(groups.unique())],
        ordered=True,
    )
    group_indices = list(groups.groupby(groups, sort=True).groups.values())
    rng = np.random.default_rng(random_seed)
    base_n = sample_size // len(group_indices)
    selected = []
    for indices in group_indices:
        selected.extend(rng.choice(
            np.asarray(indices), size=min(base_n, len(indices)), replace=False
        ).tolist())

    # 分位组不足或较小时，从未选中的行补足目标数量。
    remaining_n = sample_size - len(selected)
    if remaining_n:
        remaining_indices = np.setdiff1d(np.arange(len(pool)), selected)
        selected.extend(rng.choice(remaining_indices, size=remaining_n, replace=False).tolist())
    rng.shuffle(selected)
    sampled_df = pool.iloc[selected].copy().reset_index(drop=True)
    sampled_lengths = lengths.iloc[selected]
    print("Valid texts:", len(pool))
    print("Length groups:", len(group_indices))
    print("Actual sample size:", len(sampled_df))
    print("Length measure:", "whitespace word count" if length_fn is None else "custom length_fn")
    print("Sample length min / mean / max:",
          sampled_lengths.min(), sampled_lengths.mean(), sampled_lengths.max())
    return (sampled_df, pool) if return_grouped else sampled_df


In [17]:
from transformers import (
    AutoTokenizer,
    AutoModel,
)

def lmdfit_compute_embeddings(
    texts,
    model_name,
    batch_size=BATCH_SIZE,
    max_token=MAX_TOKEN,
):
    """Reproduce Riyadi/Agus LMDFit document embeddings.

    Each document is represented by the arithmetic mean of every position
    in ``last_hidden_state``. This follows the released Agus implementation
    exactly; padding positions are therefore included in the mean.
    """
    tokenizer = AutoTokenizer.from_pretrained(
        model_name,
        use_fast=True,
    )
    model = AutoModel.from_pretrained(model_name)

    if not torch.cuda.is_available():
        raise RuntimeError(
            "CUDA unavailable: select the CUDA-enabled Python environment"
        )
    device = torch.device(DEVICE)
    print("GPU:", torch.cuda.get_device_name(device), flush=True)
    model = model.to(device).eval()

    all_embeddings = []
    for start_index in range(0, len(texts), batch_size):
        batch_texts = texts[start_index:start_index + batch_size]
        inputs = tokenizer(
            batch_texts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=max_token,
        )
        inputs = {key: value.to(device) for key, value in inputs.items()}

        with torch.inference_mode():
            outputs = model(**inputs)
            # Riyadi/Agus baseline: ordinary mean over the sequence axis.
            # Shape changes from [batch, sequence, hidden] to [batch, hidden].
            # No attention-mask multiplication is applied.
            batch_embeddings = outputs.last_hidden_state.mean(dim=1)

        all_embeddings.append(batch_embeddings.cpu())

    embeddings = torch.cat(all_embeddings, dim=0)
    print("Model:", model_name)
    print("Device:", device)
    print("Pooling: unmasked last_hidden_state mean (Riyadi/Agus baseline)")
    print("Embedding shape:", tuple(embeddings.shape))

    del model
    torch.cuda.empty_cache()
    return embeddings


In [18]:
def riyadi_cosine_similarity(embeddings):
    """Return cosine scores for every unique unordered document pair (i < j).

    Riyadi LMDFit does not use document labels. For n sampled documents,
    the output contains n * (n - 1) / 2 cosine-similarity values.
    """
    if not torch.cuda.is_available():
        raise RuntimeError(
            "CUDA unavailable: cosine similarity requires the CUDA-enabled environment"
        )
    if embeddings.ndim != 2:
        raise ValueError("embeddings must have shape [n_documents, hidden_size].")

    device = torch.device(DEVICE)
    normalized_embeddings = F.normalize(
        embeddings.detach().to(device=device, dtype=torch.float32),
        p=2,
        dim=1,
    )

    similarity_matrix = (
        normalized_embeddings @ normalized_embeddings.T
    ).clamp(min=-1.0, max=1.0)

    number_of_documents = similarity_matrix.shape[0]
    upper_triangle_indices = torch.triu_indices(
        number_of_documents,
        number_of_documents,
        offset=1,
        device=device,
    )
    pair_scores = similarity_matrix[
        upper_triangle_indices[0],
        upper_triangle_indices[1],
    ]

    expected_pairs = number_of_documents * (number_of_documents - 1) // 2
    if pair_scores.numel() != expected_pairs:
        raise RuntimeError(
            f"Expected {expected_pairs} unique pairs, obtained {pair_scores.numel()}."
        )

    print("Documents:", number_of_documents)
    print("Unique document pairs:", pair_scores.numel())

    return pair_scores.cpu().numpy()


In [19]:
def riyadi_distribution(pair_scores):
    """Preserve all pairwise cosine scores as the LMDFit distribution.

    Riyadi/Agus computes distribution statistics directly from the raw
    pairwise scores. No label grouping, quantile grouping, or histogram
    binning is applied before calculating the mean and skewness.
    """
    pair_scores = np.asarray(pair_scores, dtype=float)

    if pair_scores.ndim != 1:
        raise ValueError("pair_scores must be a one-dimensional array.")
    if pair_scores.size == 0:
        raise ValueError("pair_scores must not be empty.")
    if not np.isfinite(pair_scores).all():
        raise ValueError("pair_scores contains NaN or infinite values.")

    distribution = pd.Series(
        pair_scores.copy(),
        name="Similarity Score",
    )

    print("Cosine similarity distribution summary:")
    print(distribution.describe().round(4))

    return distribution


In [20]:
def riyadi_distribution_metrics(distribution):
    """Calculate the two LMDFit features from raw cosine scores."""
    values = np.asarray(distribution, dtype=float)

    if values.ndim != 1:
        raise ValueError("distribution must be one-dimensional.")
    if values.size < 3:
        raise ValueError("distribution must contain at least three scores.")
    if not np.isfinite(values).all():
        raise ValueError("distribution contains NaN or infinite values.")

    cosine_mean = float(values.mean())
    fisher_pearson_skewness = float(
        skew(values, bias=False)
    )

    metrics = {
        "cosine_mean": cosine_mean,
        "fisher_pearson_skewness": fisher_pearson_skewness,
        "n_pairs": int(values.size),
    }

    print("Cosine similarity mean:", round(cosine_mean, 6))
    print(
        "Adjusted Fisher-Pearson skewness:",
        round(fisher_pearson_skewness, 6),
    )

    return metrics


In [21]:
def riyadi_run_one_model(
    texts,
    model_name,
    batch_size=BATCH_SIZE,
    max_token=MAX_TOKEN,
):
    """Run the complete LMDFit feature-extraction pipeline for one model."""
    embeddings = lmdfit_compute_embeddings(
        texts=texts,
        model_name=model_name,
        batch_size=batch_size,
        max_token=max_token,
    )
    pair_scores = riyadi_cosine_similarity(
        embeddings
    )
    distribution = riyadi_distribution(
        pair_scores
    )
    metrics = riyadi_distribution_metrics(
        distribution
    )

    return {
        "model": model_name,
        **metrics,
        "distribution": distribution,
    }


In [22]:
def riyadi_select_more_fit_models(
    model_metrics,
    random_seed=RANDOM_SEED,
):
    """Cluster models using LMDFit mean and skewness features.

    The cluster whose average skewness is closer to zero is selected as the
    more-fit cluster, following the Riyadi/Agus decision rule.
    """
    metrics_df = pd.DataFrame(model_metrics).copy()
    required_columns = {
        "model",
        "cosine_mean",
        "fisher_pearson_skewness",
    }
    missing_columns = required_columns - set(metrics_df.columns)
    if missing_columns:
        raise ValueError(
            f"Missing columns: {sorted(missing_columns)}"
        )
    if len(metrics_df) < 2:
        raise ValueError("At least two models are required for two clusters.")
    if metrics_df["model"].duplicated().any():
        raise ValueError("Each model must appear exactly once.")

    feature_columns = [
        "fisher_pearson_skewness",
        "cosine_mean",
    ]
    features = metrics_df[feature_columns].to_numpy(dtype=float)
    if not np.isfinite(features).all():
        raise ValueError("LMDFit features contain NaN or infinite values.")

    clustering = KMeans(
        n_clusters=2,
        random_state=random_seed,
        n_init=10,
    )
    metrics_df["cluster"] = clustering.fit_predict(features)

    cluster_summary = (
        metrics_df
        .groupby("cluster", as_index=False)
        .agg(
            model_count=("model", "size"),
            average_skewness=("fisher_pearson_skewness", "mean"),
            average_cosine_mean=("cosine_mean", "mean"),
        )
    )
    cluster_summary["absolute_average_skewness"] = (
        cluster_summary["average_skewness"].abs()
    )
    more_fit_cluster = int(
        cluster_summary
        .sort_values(
            ["absolute_average_skewness", "cluster"],
            ascending=True,
        )
        .iloc[0]["cluster"]
    )
    metrics_df["is_more_fit"] = (
        metrics_df["cluster"] == more_fit_cluster
    )
    more_fit_models = (
        metrics_df.loc[metrics_df["is_more_fit"], "model"].tolist()
    )

    print("Cluster summary:")
    display(cluster_summary)
    print("More-fit cluster:", more_fit_cluster)
    print("More-fit models:", more_fit_models)

    return {
        "model_assignments": metrics_df,
        "cluster_summary": cluster_summary,
        "more_fit_cluster": more_fit_cluster,
        "more_fit_models": more_fit_models,
        "kmeans": clustering,
    }


### 2.0 Baseline Experiment

In [23]:
MODELS = [
    "google-bert/bert-base-uncased",
    "allenai/scibert_scivocab_uncased",
    "nlpaueb/legal-bert-base-uncased",
    "ahmedrachid/FinancialBERT",
    "Lianglab/PharmBERT-uncased",
    "recobo/agriculture-bert-uncased",
    "recobo/chemical-bert-uncased",
]

In [24]:
from pathlib import Path
import numpy as np
import pandas as pd
from datasets import load_from_disk
from IPython.display import display

BASE_DIR = Path.cwd()
if (BASE_DIR / "Standardizing").is_dir():
    BASE_DIR = BASE_DIR / "Standardizing"
if not (BASE_DIR / "Financial" / "financial_original.csv").exists():
    raise FileNotFoundError(
        "Run this Notebook from the GR project root or Standardizing directory."
    )

pd.set_option("display.max_colwidth", 100)
print("Data directory:", BASE_DIR)


def to_frame(frame, dataset_name, text_col, source_split, id_col=None):
    """Standardize only the document identity and text needed by LMDFit."""
    result = pd.DataFrame(index=frame.index)
    # The split prefix keeps locally repeated source IDs globally unique.
    ids = frame[id_col].astype(str) if id_col else frame.index.astype(str)
    result["document_id"] = [
        f"{dataset_name}:{source_split}:{value}"
        for value in ids
    ]
    result["text"] = frame[text_col]
    result["source_split"] = source_split
    return result.reset_index(drop=True)


raw_frames = {}
source_nan_rows = []


def check_source(name, frame):
    """Report NaN counts in every original source column."""
    print(f"\n{name}: shape={frame.shape}")
    missing = frame.isna().sum()
    display(missing.rename("NaN_count").to_frame())
    for column, count in missing.items():
        source_nan_rows.append(
            {"source": name, "column": column, "NaN_count": int(count)}
        )


financial_source = pd.read_csv(
    BASE_DIR / "Financial" / "financial_original.csv"
)
check_source("Financial", financial_source)
raw_frames["financial"] = to_frame(
    financial_source, "financial", "sentence", "original"
)

envdata_source = pd.read_csv(
    BASE_DIR / "EnvData" / "environmental_claims_dataset_original.csv"
)
check_source("EnvData", envdata_source)
raw_frames["envdata"] = to_frame(
    envdata_source, "envdata", "text", "original"
)

agnews_source = load_from_disk(
    str(BASE_DIR / "ACGNews" / "data" / "ag_news")
)
agnews_parts = []
for split, dataset in agnews_source.items():
    frame = dataset.to_pandas()
    check_source(f"ACGNews/{split}", frame)
    agnews_parts.append(
        to_frame(frame, "acgnews", "text", split)
    )
raw_frames["acgnews"] = pd.concat(agnews_parts, ignore_index=True)

ecthr_parts = []
for split, filename in [
    ("train", "train.jsonl"),
    ("validation", "dev.jsonl"),
    ("test", "test.jsonl"),
]:
    frame = pd.read_json(
        BASE_DIR / "ECtHR" / "data" / "ecthr_cases_raw" / filename,
        lines=True,
    )
    check_source(f"ECtHR/{split}", frame)
    # Preserve missing facts as missing instead of the literal string "None".
    frame["joined_text"] = frame["facts"].map(
        lambda facts: " ".join(
            str(part) for part in facts if pd.notna(part)
        )
        if isinstance(facts, (list, tuple, np.ndarray))
        else None
    )
    ecthr_parts.append(
        to_frame(
            frame,
            "ecthr",
            "joined_text",
            split,
            "case_id",
        )
    )
raw_frames["ecthr"] = pd.concat(ecthr_parts, ignore_index=True)

# The same Ohsumed document can occur in multiple category directories.
# LMDFit ignores those category names, so documents are consolidated by ID.
ohsumed_documents = {}
for category_dir in sorted(
    (BASE_DIR / "Ohsumed" / "data" / "ohsumed-all").glob("C*")
):
    if not category_dir.is_dir():
        continue
    for path in sorted(category_dir.iterdir()):
        if not path.is_file():
            continue
        text = path.read_text(
            encoding="latin-1",
            errors="replace",
        ).strip()
        row = ohsumed_documents.setdefault(
            path.name,
            {"id": path.name, "text": text},
        )
        if row["text"] != text:
            raise ValueError(
                f"Ohsumed document ID has conflicting text: {path.name}"
            )

ohsumed_source = pd.DataFrame(ohsumed_documents.values())
if ohsumed_source.empty:
    raise ValueError("No Ohsumed documents were found.")
check_source("Ohsumed/by_document_id", ohsumed_source)
raw_frames["ohsumed"] = to_frame(
    ohsumed_source,
    "ohsumed",
    "text",
    "original",
    "id",
)

rheology_parts = []
for split in ["train", "dev"]:
    frame = pd.read_json(
        BASE_DIR / "Rheology" / "data" / "RHEOLOGY" / f"{split}.json",
        lines=True,
    )
    check_source(f"Rheology/{split}", frame)
    rheology_parts.append(
        to_frame(frame, "rheology", "sentence", split, "id")
    )
raw_frames["rheology"] = pd.concat(rheology_parts, ignore_index=True)

arxiv_source = pd.read_csv(
    BASE_DIR / "arXiv" / "arxiv_data.csv"
)
check_source("arXiv", arxiv_source)
raw_frames["arxiv"] = to_frame(
    arxiv_source,
    "arxiv",
    "summaries",
    "original",
)


Data directory: /home/amyliu/Desktop/GR/Standardizing

Financial: shape=(4846, 2)


,NaN_count
sentence,0
label,0



EnvData: shape=(2647, 2)


,NaN_count
text,0
label,0



ACGNews/train: shape=(120000, 2)


,NaN_count
text,0
label,0


... (12 outputs left out)


Rheology/dev: shape=(403, 3)


,NaN_count
id,0
sentence,0
label,0



arXiv: shape=(51774, 3)


,NaN_count
titles,0
summaries,0
terms,0


In [25]:
def clean_dataset(frame):
    """Clean only text fields; LMDFit does not inspect class labels."""
    data = frame.copy()
    missing_text = data["text"].isna()
    data["text"] = (
        data["text"]
        .astype("string")
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )
    blank_text = data["text"].fillna("").eq("") & ~missing_text
    invalid = missing_text | blank_text

    rejected = data.loc[invalid].copy()
    valid = data.loc[~invalid].copy()

    grouped = valid.groupby("text", sort=False, dropna=False)
    text_sizes = grouped.size()
    clean = grouped.agg(
        document_id=("document_id", "first"),
        source_splits=("source_split", lambda values: sorted(set(values))),
        source_document_ids=("document_id", list),
    ).reset_index()
    clean = clean[
        [
            "document_id",
            "text",
            "source_splits",
            "source_document_ids",
        ]
    ].reset_index(drop=True)

    assert clean["document_id"].is_unique
    assert clean["text"].is_unique
    assert clean["text"].notna().all()
    assert clean["text"].str.len().gt(0).all()

    duplicate_rows_merged = len(valid) - len(clean)
    summary = {
        "input_rows": len(frame),
        "missing_text": int(missing_text.sum()),
        "blank_text": int(blank_text.sum()),
        "rejected_rows": len(rejected),
        "duplicate_text_groups": int(text_sizes.gt(1).sum()),
        "duplicate_excess_rows": int(
            (text_sizes - 1).clip(lower=0).sum()
        ),
        "duplicate_rows_merged": duplicate_rows_merged,
        "final_rows": len(clean),
    }
    assert len(frame) == len(rejected) + duplicate_rows_merged + len(clean)
    return clean, rejected, summary


clean_frames = {}
df_rejected = {}
cleaning_rows = []
for name, raw in raw_frames.items():
    clean, rejected, summary = clean_dataset(raw)
    clean_frames[name] = clean
    df_rejected[name] = rejected
    cleaning_rows.append({"dataset": name, **summary})


df_financial = clean_frames["financial"]
df_envdata = clean_frames["envdata"]
df_acgnews = clean_frames["acgnews"]
df_ecthr = clean_frames["ecthr"]
df_ohsumed = clean_frames["ohsumed"]
df_rheology = clean_frames["rheology"]
df_arxiv = clean_frames["arxiv"]

df_source_nan_summary = pd.DataFrame(source_nan_rows)
df_cleaning_summary = pd.DataFrame(cleaning_rows)

print("Text-only LMDFit cleaning summary:")
display(df_cleaning_summary)
print(
    "No labels were parsed, filtered, encoded, validated, or used for deduplication."
)


Text-only LMDFit cleaning summary:


,dataset,input_rows,missing_text,blank_text,rejected_rows,duplicate_text_groups,duplicate_excess_rows,duplicate_rows_merged,final_rows
0,financial,4846,0,0,0,8,8,8,4838
1,envdata,2647,0,0,0,0,0,0,2647
2,acgnews,127600,0,0,0,83,83,83,127517
3,ecthr,11000,0,0,0,1,1,1,10999
4,ohsumed,34389,0,0,0,3,3,3,34386
5,rheology,2015,0,0,0,9,9,9,2006
6,arxiv,51774,0,0,0,10401,12795,12795,38979


No labels were parsed, filtered, encoded, validated, or used for deduplication.


In [26]:
prediction_datasets = {
    "Financial": df_financial,
    "EnvData": df_envdata,
    "ACGNews": df_acgnews,
    "ECtHR": df_ecthr,
    "Ohsumed": df_ohsumed,
    "Rheology": df_rheology,
    "arXiv": df_arxiv,
}

dataset_preparation_summary = pd.DataFrame(
    [
        {
            "dataset": dataset_name,
            "documents_after_text_cleaning": len(frame),
            "unique_document_ids": frame["document_id"].nunique(),
            "unique_texts": frame["text"].nunique(),
            "missing_texts": int(frame["text"].isna().sum()),
            "blank_texts": int(frame["text"].fillna("").eq("").sum()),
        }
        for dataset_name, frame in prediction_datasets.items()
    ]
)

assert dataset_preparation_summary["documents_after_text_cleaning"].gt(0).all()
assert dataset_preparation_summary["documents_after_text_cleaning"].eq(
    dataset_preparation_summary["unique_document_ids"]
).all()
assert dataset_preparation_summary["documents_after_text_cleaning"].eq(
    dataset_preparation_summary["unique_texts"]
).all()
assert dataset_preparation_summary["missing_texts"].eq(0).all()
assert dataset_preparation_summary["blank_texts"].eq(0).all()

print("Dataset preparation summary (labels are intentionally absent):")
display(dataset_preparation_summary)


Dataset preparation summary (labels are intentionally absent):


,dataset,documents_after_text_cleaning,unique_document_ids,unique_texts,missing_texts,blank_texts
0,Financial,4838,4838,4838,0,0
1,EnvData,2647,2647,2647,0,0
2,ACGNews,127517,127517,127517,0,0
3,ECtHR,10999,10999,10999,0,0
4,Ohsumed,34386,34386,34386,0,0
5,Rheology,2006,2006,2006,0,0
6,arXiv,38979,38979,38979,0,0


### Uncertainty Test and Pearson Correlation for Baseline

In [27]:
UNCERTAINTY_REPETITIONS = 10
UNCERTAINTY_MASTER_SEED = 42
UNCERTAINTY_SAMPLE_SIZE = SAMPLE_SIZE
UNCERTAINTY_NUM_QUANTILES = NUM_QUANTILES


In [28]:
# RIYADI_UNCERTAINTY10_GENERATED_SECTION: setup and reusable experiment helpers
from transformers import AutoModel, AutoTokenizer
import gc

RIYADI_UNCERTAINTY_OUTPUT_DIR = (
    BASE_DIR / "results" / "riyadi_baseline_uncertainty10_reproduced"
)
RIYADI_UNCERTAINTY_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RIYADI_SAMPLE_DIR = (
    BASE_DIR / "results" / "rank_prediction_uncertainty10_label_first"
)
RIYADI_RAW_METRICS_PATH = (
    RIYADI_UNCERTAINTY_OUTPUT_DIR / "baseline_metrics_10runs.csv"
)
RIYADI_UNCERTAINTY_SUMMARY_PATH = (
    RIYADI_UNCERTAINTY_OUTPUT_DIR / "metric_uncertainty_summary.csv"
)
RIYADI_MODEL_MEANS_PATH = (
    RIYADI_UNCERTAINTY_OUTPUT_DIR / "model_metrics_10run_means.csv"
)
RIYADI_PEARSON_PATH = (
    RIYADI_UNCERTAINTY_OUTPUT_DIR / "pearson_correlations.csv"
)


def riyadi_load_saved_samples(dataset_name, dataset_frame):
    """Load the exact ten paired samples previously used by our method."""
    sample_path = RIYADI_SAMPLE_DIR / (
        dataset_name.lower().replace(" ", "_") + "_sampled_documents.csv"
    )
    saved = pd.read_csv(sample_path, dtype={"document_id": str})
    required = {"repetition", "random_seed", "sample_position", "document_id"}
    missing_columns = required - set(saved.columns)
    if missing_columns:
        raise ValueError(
            f"{dataset_name} sample file is missing {sorted(missing_columns)}"
        )

    lookup = (
        dataset_frame
        .assign(document_id=dataset_frame["document_id"].astype(str))
        .set_index("document_id", drop=False)
    )
    samples = {}
    for repetition, rows in saved.groupby("repetition", sort=True):
        rows = rows.sort_values("sample_position")
        missing_ids = sorted(set(rows["document_id"]) - set(lookup.index))
        if missing_ids:
            raise ValueError(
                f"{dataset_name} repetition {repetition} has missing IDs: "
                f"{missing_ids[:3]}"
            )
        sample = lookup.loc[rows["document_id"].tolist()].reset_index(drop=True)
        if len(sample) != UNCERTAINTY_SAMPLE_SIZE:
            raise ValueError(
                f"{dataset_name} repetition {repetition} has {len(sample)} rows"
            )
        if sample["document_id"].nunique() != UNCERTAINTY_SAMPLE_SIZE:
            raise ValueError(
                f"{dataset_name} repetition {repetition} repeats document IDs"
            )
        samples[int(repetition)] = {
            "random_seed": int(rows["random_seed"].iloc[0]),
            "frame": sample,
        }

    expected_repetitions = set(range(1, UNCERTAINTY_REPETITIONS + 1))
    if set(samples) != expected_repetitions:
        raise ValueError(
            f"{dataset_name} repetitions are {sorted(samples)}, "
            f"expected {sorted(expected_repetitions)}"
        )
    return samples


def riyadi_embeddings_with_loaded_model(
    texts,
    model,
    tokenizer,
    batch_size=BATCH_SIZE,
    max_token=MAX_TOKEN,
):
    """Efficient form of lmdfit_compute_embeddings for a reused loaded model."""
    batches = []
    for start in range(0, len(texts), batch_size):
        inputs = tokenizer(
            texts[start:start + batch_size],
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=max_token,
        )
        inputs = {key: value.to(DEVICE) for key, value in inputs.items()}
        with torch.inference_mode():
            # Same unmasked final-layer mean as the reproduced Riyadi/Agus architecture.
            embeddings = model(**inputs).last_hidden_state.mean(dim=1)
        batches.append(embeddings.cpu())
    return torch.cat(batches, dim=0)


In [29]:
# RIYADI_UNCERTAINTY10_GENERATED_SECTION: run or resume 7 datasets x 10 repetitions x 7 models
if not torch.cuda.is_available():
    raise RuntimeError("CUDA unavailable: select the RTX 5080-enabled kernel")

uncertainty_samples = {
    dataset_name: riyadi_load_saved_samples(dataset_name, dataset_frame)
    for dataset_name, dataset_frame in prediction_datasets.items()
}

if RIYADI_RAW_METRICS_PATH.exists():
    uncertainty_rows = pd.read_csv(RIYADI_RAW_METRICS_PATH).to_dict("records")
else:
    uncertainty_rows = []

completed_keys = {
    (row["dataset"], int(row["repetition"]), row["model"])
    for row in uncertainty_rows
}
expected_rows = (
    len(prediction_datasets)
    * UNCERTAINTY_REPETITIONS
    * len(MODELS)
)
print(f"Completed before this run: {len(completed_keys)}/{expected_rows}")

for model_number, model_name in enumerate(MODELS, start=1):
    missing_runs = [
        (dataset_name, repetition)
        for dataset_name in prediction_datasets
        for repetition in range(1, UNCERTAINTY_REPETITIONS + 1)
        if (dataset_name, repetition, model_name) not in completed_keys
    ]
    if not missing_runs:
        print(f"Skip completed model {model_number}/{len(MODELS)}: {model_name}")
        continue

    print("=" * 88)
    print(f"Model {model_number}/{len(MODELS)}: {model_name}")
    tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
    model = AutoModel.from_pretrained(model_name).to(DEVICE).eval()

    for dataset_name, repetition in missing_runs:
        run_started = time.perf_counter()
        sample_info = uncertainty_samples[dataset_name][repetition]
        texts = (
            sample_info["frame"][TEXT_COL]
            .astype(str)
            .tolist()
        )
        embeddings = riyadi_embeddings_with_loaded_model(
            texts=texts,
            model=model,
            tokenizer=tokenizer,
        )
        pair_scores = riyadi_cosine_similarity(embeddings)
        distribution = riyadi_distribution(pair_scores)
        metrics = riyadi_distribution_metrics(distribution)

        row = {
            "dataset": dataset_name,
            "repetition": repetition,
            "random_seed": sample_info["random_seed"],
            "model": model_name,
            "cosine_mean": metrics["cosine_mean"],
            "fisher_pearson_skewness": metrics["fisher_pearson_skewness"],
            "n_pairs": metrics["n_pairs"],
            "sample_size": UNCERTAINTY_SAMPLE_SIZE,
            "max_token": MAX_TOKEN,
            "batch_size": BATCH_SIZE,
            "pooling": "unmasked_last_hidden_state_mean",
            "skew_bias": False,
            "elapsed_seconds": time.perf_counter() - run_started,
        }
        uncertainty_rows.append(row)
        completed_keys.add((dataset_name, repetition, model_name))
        (
            pd.DataFrame(uncertainty_rows)
            .sort_values(["dataset", "repetition", "model"])
            .to_csv(RIYADI_RAW_METRICS_PATH, index=False)
        )
        print(
            f"{dataset_name:10s} repetition={repetition:2d} "
            f"mean={metrics['cosine_mean']:.6f} "
            f"skewness={metrics['fisher_pearson_skewness']:.6f} "
            f"elapsed={row['elapsed_seconds']:.2f}s"
        )

    del model, tokenizer
    gc.collect()
    torch.cuda.empty_cache()

baseline_uncertainty_results = (
    pd.read_csv(RIYADI_RAW_METRICS_PATH)
    .sort_values(["dataset", "repetition", "model"])
    .reset_index(drop=True)
)
if len(baseline_uncertainty_results) != expected_rows:
    raise RuntimeError(
        f"Expected {expected_rows} completed rows, obtained "
        f"{len(baseline_uncertainty_results)}"
    )
if baseline_uncertainty_results[
    ["dataset", "repetition", "model"]
].duplicated().any():
    raise RuntimeError("Duplicate dataset/repetition/model result rows found")
if not baseline_uncertainty_results["n_pairs"].eq(
    UNCERTAINTY_SAMPLE_SIZE * (UNCERTAINTY_SAMPLE_SIZE - 1) // 2
).all():
    raise RuntimeError("Unexpected number of document pairs")

print(f"Completed: {len(baseline_uncertainty_results)}/{expected_rows}")
print("Saved:", RIYADI_RAW_METRICS_PATH)
display(baseline_uncertainty_results.tail(10))


Completed before this run: 0/490
Model 1/7: google-bert/bert-base-uncased


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: google-bert/bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Documents: 200
Unique document pairs: 19900
Cosine similarity distribution summary:
count    19900.0000
mean         0.6870
std          0.0812
min          0.3793
25%          0.6387
50%          0.6933
75%          0.7412
max          0.9552
Name: Similarity Score, dtype: float64
Cosine similarity mean: 0.687021
Adjusted Fisher-Pearson skewness: -0.385921
Financial  repetition= 1 mean=0.687021 skewness=-0.385921 elapsed=0.25s
Documents: 200
Unique document pairs: 19900
Cosine similarity distribution summary:
count    19900.0000
mean         0.7087
std          0.0804
min          0.3576
25%          0.6594
50%          0.7120
75%          0.7630
max          0.9561
Name: Similarity Score, dtype: float64
Cosine similarity mean: 0.708748
Adjusted Fisher-Pearson skewness: -0.338189
Financial  repetition= 2 mean=0.708748 skewness=-0.338189 elapsed=0.12s
Documents: 200
Unique document pairs: 19900
Cosine similarity distribution summary:
count    19900.0000
mean         0.6939
std         

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: allenai/scibert_scivocab_uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


... (13 outputs left out)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertModel LOAD REPORT from: recobo/chemical-bert-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
pooler.dense.weight                        | MISSING    | 
pooler.dense.bias                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Documents: 200
Unique document pairs: 19900
Cosine similarity distribution summary:
count    19900.0000
mean         0.8219
std          0.0602
min          0.4949
25%          0.7870
50%          0.8292
75%          0.8649
max          0.9650
Name: Similarity Score, dtype: float64
Cosine similarity mean: 0.821869
Adjusted Fisher-Pearson skewness: -1.001942
Financial  repetition= 1 mean=0.821869 skewness=-1.001942 elapsed=0.12s
Documents: 200
Unique document pairs: 19900
Cosine similarity distribution summary:
count    19900.0000
mean         0.8268
std          0.0674
min          0.4399
25%          0.7943
50%          0.8388
75%          0.8737
max          0.9675
Name: Similarity Score, dtype: float64
Cosine similarity mean: 0.826834
Adjusted Fisher-Pearson skewness: -1.326608
Financial  repetition= 2 mean=0.826834 skewness=-1.326608 elapsed=0.12s
Documents: 200
Unique document pairs: 19900
Cosine similarity distribution summary:
count    19900.0000
mean         0.8243
std         

,dataset,repetition,random_seed,model,cosine_mean,fisher_pearson_skewness,n_pairs,sample_size,max_token,batch_size,pooling,skew_bias,elapsed_seconds
480,arXiv,9,865305066,nlpaueb/legal-bert-base-uncased,0.959689,-1.780823,19900,200,512,64,unmasked_last_hidden_state_mean,False,0.697751
481,arXiv,9,865305066,recobo/agriculture-bert-uncased,0.907071,-0.715662,19900,200,512,64,unmasked_last_hidden_state_mean,False,0.621364
482,arXiv,9,865305066,recobo/chemical-bert-uncased,0.934721,-1.820237,19900,200,512,64,unmasked_last_hidden_state_mean,False,0.619020
483,arXiv,10,404488628,Lianglab/PharmBERT-uncased,0.902675,-1.360222,19900,200,512,64,unmasked_last_hidden_state_mean,False,0.712964
484,arXiv,10,404488628,ahmedrachid/FinancialBERT,0.913917,-2.723690,19900,200,512,64,unmasked_last_hidden_state_mean,False,0.739620
485,arXiv,10,404488628,allenai/scibert_scivocab_uncased,0.878270,-1.487914,19900,200,512,64,unmasked_last_hidden_state_mean,False,0.552886
486,arXiv,10,404488628,google-bert/bert-base-uncased,0.895002,-2.074131,19900,200,512,64,unmasked_last_hidden_state_mean,False,0.714422
487,arXiv,10,404488628,nlpaueb/legal-bert-base-uncased,0.959914,-2.127797,19900,200,512,64,unmasked_last_hidden_state_mean,False,0.751460
488,arXiv,10,404488628,recobo/agriculture-bert-uncased,0.905693,-1.060346,19900,200,512,64,unmasked_last_hidden_state_mean,False,0.657603
489,arXiv,10,404488628,recobo/chemical-bert-uncased,0.936901,-1.827353,19900,200,512,64,unmasked_last_hidden_state_mean,False,0.658817


In [30]:
# RIYADI_UNCERTAINTY10_GENERATED_SECTION: uncertainty summaries for mean and skewness
metric_columns = [
    "cosine_mean",
    "fisher_pearson_skewness",
]

uncertainty_long = baseline_uncertainty_results.melt(
    id_vars=["dataset", "model", "repetition", "random_seed"],
    value_vars=metric_columns,
    var_name="metric",
    value_name="value",
)

metric_uncertainty_summary = (
    uncertainty_long
    .groupby(["dataset", "model", "metric"], as_index=False)
    .agg(
        mean=("value", "mean"),
        sd=("value", "std"),
        minimum=("value", "min"),
        maximum=("value", "max"),
        repetitions=("value", "size"),
    )
)
metric_uncertainty_summary["sem"] = (
    metric_uncertainty_summary["sd"]
    / np.sqrt(metric_uncertainty_summary["repetitions"])
)
# Descriptive 95% t interval for the ten repeated samples.
t_critical = stats.t.ppf(
    0.975,
    metric_uncertainty_summary["repetitions"] - 1,
)
metric_uncertainty_summary["ci95_low"] = (
    metric_uncertainty_summary["mean"]
    - t_critical * metric_uncertainty_summary["sem"]
)
metric_uncertainty_summary["ci95_high"] = (
    metric_uncertainty_summary["mean"]
    + t_critical * metric_uncertainty_summary["sem"]
)
metric_uncertainty_summary.to_csv(
    RIYADI_UNCERTAINTY_SUMMARY_PATH,
    index=False,
)

model_metrics_10run_means = (
    baseline_uncertainty_results
    .groupby(["dataset", "model"], as_index=False)
    .agg(
        cosine_mean=("cosine_mean", "mean"),
        cosine_mean_sd=("cosine_mean", "std"),
        fisher_pearson_skewness=("fisher_pearson_skewness", "mean"),
        fisher_pearson_skewness_sd=("fisher_pearson_skewness", "std"),
        repetitions=("repetition", "nunique"),
    )
)
model_metrics_10run_means.to_csv(
    RIYADI_MODEL_MEANS_PATH,
    index=False,
)

print("Metric uncertainty summary:")
display(metric_uncertainty_summary)
print("Saved:", RIYADI_UNCERTAINTY_SUMMARY_PATH)
print("Saved:", RIYADI_MODEL_MEANS_PATH)


Metric uncertainty summary:


,dataset,model,metric,mean,sd,minimum,maximum,repetitions,sem,ci95_low,ci95_high
0,ACGNews,Lianglab/PharmBERT-uncased,cosine_mean,0.780517,0.004918,0.772109,0.788897,10,0.001555,0.776998,0.784035
1,ACGNews,Lianglab/PharmBERT-uncased,fisher_pearson_skewness,-0.506087,0.155223,-0.850638,-0.356731,10,0.049086,-0.617127,-0.395047
2,ACGNews,ahmedrachid/FinancialBERT,cosine_mean,0.821092,0.004528,0.810689,0.826053,10,0.001432,0.817853,0.824331
3,ACGNews,ahmedrachid/FinancialBERT,fisher_pearson_skewness,-0.711830,0.136674,-0.962051,-0.513741,10,0.043220,-0.809601,-0.614059
4,ACGNews,allenai/scibert_scivocab_uncased,cosine_mean,0.845747,0.003058,0.841793,0.850211,10,0.000967,0.843560,0.847934
...,...,...,...,...,...,...,...,...,...,...,...
93,arXiv,nlpaueb/legal-bert-base-uncased,fisher_pearson_skewness,-2.359455,0.472667,-3.266961,-1.780823,10,0.149470,-2.697580,-2.021330
94,arXiv,recobo/agriculture-bert-uncased,cosine_mean,0.905038,0.001976,0.900820,0.907071,10,0.000625,0.903624,0.906451
95,arXiv,recobo/agriculture-bert-uncased,fisher_pearson_skewness,-0.950175,0.259482,-1.398885,-0.553986,10,0.082056,-1.135797,-0.764552
96,arXiv,recobo/chemical-bert-uncased,cosine_mean,0.936189,0.001716,0.933536,0.938401,10,0.000543,0.934961,0.937416


Saved: /home/amyliu/Desktop/GR/Standardizing/results/riyadi_baseline_uncertainty10_reproduced/metric_uncertainty_summary.csv
Saved: /home/amyliu/Desktop/GR/Standardizing/results/riyadi_baseline_uncertainty10_reproduced/model_metrics_10run_means.csv


In [33]:
# RIYADI_UNCERTAINTY10_GENERATED_SECTION: signed Pearson correlations only
fine_tuning_summary_path = (
    BASE_DIR / "results" / "finetuning_8epochs_5fold" / "model_summary.csv"
)
fine_tuning_summary = pd.read_csv(fine_tuning_summary_path)
f1_columns = ["micro_f1_mean", "macro_f1_mean"]

comparison_data = model_metrics_10run_means.merge(
    fine_tuning_summary[
        ["dataset", "model", *f1_columns]
    ],
    on=["dataset", "model"],
    how="left",
    validate="one_to_one",
)
if comparison_data[f1_columns].isna().any().any():
    missing = comparison_data.loc[
        comparison_data[f1_columns].isna().any(axis=1),
        ["dataset", "model"],
    ]
    raise ValueError(f"Missing fine-tuning F1 rows:\n{missing}")

pearson_rows = []
for dataset_name, group in comparison_data.groupby("dataset", sort=False):
    if len(group) != len(MODELS):
        raise ValueError(
            f"{dataset_name} has {len(group)} models, expected {len(MODELS)}"
        )
    for target in f1_columns:
        for metric in ["cosine_mean", "fisher_pearson_skewness"]:
            pearson_r, p_value = stats.pearsonr(
                group[metric],
                group[target],
            )
            pearson_rows.append({
                "dataset": dataset_name,
                "target": target,
                "metric": metric,
                "pearson_signed": float(pearson_r),
                "p_value": float(p_value),
                "n_models": len(group),
            })

baseline_pearson_correlations = pd.DataFrame(pearson_rows)
baseline_pearson_correlations.to_csv(
    RIYADI_PEARSON_PATH,
    index=False,
)
pearson_signed_average = (
    baseline_pearson_correlations
    .groupby(["target", "metric"], as_index=False)
    .agg(
        average_signed_pearson=("pearson_signed", "mean"),
        dataset_sd=("pearson_signed", "std"),
        datasets=("dataset", "nunique"),
    )
)

print("Signed Pearson correlations by dataset:")
display(baseline_pearson_correlations)
print("Average signed Pearson across seven datasets:")
display(pearson_signed_average)
print("Saved:", RIYADI_PEARSON_PATH)


Signed Pearson correlations by dataset:


,dataset,target,metric,pearson_signed,p_value,n_models
0,ACGNews,micro_f1_mean,cosine_mean,-0.414892,0.354679,7
1,ACGNews,micro_f1_mean,fisher_pearson_skewness,-0.006674,0.988670,7
2,ACGNews,macro_f1_mean,cosine_mean,-0.415588,0.353791,7
3,ACGNews,macro_f1_mean,fisher_pearson_skewness,-0.006579,0.988832,7
4,ECtHR,micro_f1_mean,cosine_mean,-0.710063,0.073824,7
5,ECtHR,micro_f1_mean,fisher_pearson_skewness,0.655181,0.110180,7
6,ECtHR,macro_f1_mean,cosine_mean,-0.542202,0.208635,7
7,ECtHR,macro_f1_mean,fisher_pearson_skewness,0.867460,0.011423,7
8,EnvData,micro_f1_mean,cosine_mean,-0.172296,0.711823,7
9,EnvData,micro_f1_mean,fisher_pearson_skewness,0.112659,0.809956,7


Average signed Pearson across seven datasets:


,target,metric,average_signed_pearson,dataset_sd,datasets
0,macro_f1_mean,cosine_mean,-0.371497,0.216878,7
1,macro_f1_mean,fisher_pearson_skewness,0.313122,0.353748,7
2,micro_f1_mean,cosine_mean,-0.396922,0.228332,7
3,micro_f1_mean,fisher_pearson_skewness,0.248467,0.282554,7


Saved: /home/amyliu/Desktop/GR/Standardizing/results/riyadi_baseline_uncertainty10_reproduced/pearson_correlations.csv


Table 19-style signed Pearson summary

Pearson correlations retain their direction. `Average` is the arithmetic mean of the signed coefficients across the seven datasets.


In [34]:
# RIYADI_TABLE19_PEARSON_SUMMARY: signed correlations only
pearson_results_for_table = pd.read_csv(
    RIYADI_PEARSON_PATH
)

paper_dataset_order = [
    "Financial",
    "EnvData",
    "ACGNews",
    "ECtHR",
    "Ohsumed",
    "Rheology",
    "arXiv",
]
paper_column_order = [
    ("micro_f1_mean", "cosine_mean"),
    ("micro_f1_mean", "fisher_pearson_skewness"),
    ("macro_f1_mean", "cosine_mean"),
    ("macro_f1_mean", "fisher_pearson_skewness"),
]
paper_column_labels = pd.MultiIndex.from_tuples(
    [
        ("F1 Micro", "Cosine Similarity Mean"),
        ("F1 Micro", "Fisher-Pearson Skewness"),
        ("F1 Macro", "Cosine Similarity Mean"),
        ("F1 Macro", "Fisher-Pearson Skewness"),
    ]
)

table19_pearson_signed = pearson_results_for_table.pivot(
    index="dataset",
    columns=["target", "metric"],
    values="pearson_signed",
)
table19_pearson_signed = table19_pearson_signed.loc[
    paper_dataset_order,
    paper_column_order,
]
table19_pearson_signed.columns = paper_column_labels
table19_pearson_signed.loc["Average"] = (
    table19_pearson_signed.mean(axis=0)
)
table19_pearson_signed.index.name = "Dataset"

signed_table_path = (
    RIYADI_UNCERTAINTY_OUTPUT_DIR
    / "table19_style_pearson_signed.csv"
)
table19_pearson_signed.to_csv(signed_table_path)

print("Signed Pearson correlations (direction preserved):")
display(table19_pearson_signed.round(3))
print("Saved:", signed_table_path)


Signed Pearson correlations (direction preserved):


F1 Micro                          \
          Cosine Similarity Mean Fisher-Pearson Skewness   
Dataset                                                    
Financial                 -0.140                   0.081   
EnvData                   -0.172                   0.113   
ACGNews                   -0.415                  -0.007   
ECtHR                     -0.710                   0.655   
Ohsumed                   -0.612                   0.271   
Rheology                  -0.207                   0.002   
arXiv                     -0.522                   0.624   
Average                   -0.397                   0.248   

                        F1 Macro                          
          Cosine Similarity Mean Fisher-Pearson Skewness  
Dataset                                                   
Financial                 -0.196                   0.131  
EnvData                   -0.162                   0.086  
ACGNews                   -0.416                  -0.007  
ECtHR                     -0.542                   0.867  
Ohsumed                   -0.674                   0.381  
Rheology                  -0.110                   0.015  
arXiv                     -0.501                   0.717  
Average                   -0.371                   0.313

Saved: /home/amyliu/Desktop/GR/Standardizing/results/riyadi_baseline_uncertainty10_reproduced/table19_style_pearson_signed.csv


### Table 19-style signed Spearman summary

Spearman correlations evaluate agreement between the metric ranking and the five-fold F1 ranking across the seven candidate models. Direction is preserved, and `Average` is the arithmetic mean of the signed coefficients across the seven datasets.


In [ ]:
# RIYADI_TABLE19_SPEARMAN_SUMMARY: signed Spearman correlations only
from scipy.stats import spearmanr

baseline_model_metrics_for_spearman = pd.read_csv(
    RIYADI_MODEL_MEANS_PATH
)
baseline_f1_for_spearman = pd.read_csv(
    BASE_DIR / "results" / "finetuning_8epochs_5fold" / "model_summary.csv"
)

baseline_spearman_data = baseline_model_metrics_for_spearman.merge(
    baseline_f1_for_spearman[
        ["dataset", "model", "micro_f1_mean", "macro_f1_mean"]
    ],
    on=["dataset", "model"],
    how="left",
    validate="one_to_one",
)
if baseline_spearman_data[
    ["micro_f1_mean", "macro_f1_mean"]
].isna().any().any():
    raise ValueError("Missing fine-tuning F1 values in baseline Spearman data.")

baseline_spearman_rows = []
for dataset_name, group in baseline_spearman_data.groupby("dataset", sort=False):
    if len(group) != len(MODELS):
        raise ValueError(
            f"{dataset_name} has {len(group)} models; expected {len(MODELS)}."
        )
    for target in ["micro_f1_mean", "macro_f1_mean"]:
        for metric in ["cosine_mean", "fisher_pearson_skewness"]:
            result = spearmanr(group[metric], group[target])
            baseline_spearman_rows.append({
                "dataset": dataset_name,
                "target": target,
                "metric": metric,
                "spearman_signed": float(result.statistic),
                "p_value": float(result.pvalue),
                "n_models": len(group),
            })

baseline_spearman_correlations = pd.DataFrame(
    baseline_spearman_rows
)
baseline_spearman_path = (
    RIYADI_UNCERTAINTY_OUTPUT_DIR
    / "spearman_correlations.csv"
)
baseline_spearman_correlations.to_csv(
    baseline_spearman_path,
    index=False,
)

baseline_spearman_table = baseline_spearman_correlations.pivot(
    index="dataset",
    columns=["target", "metric"],
    values="spearman_signed",
)
baseline_spearman_table = baseline_spearman_table.loc[
    paper_dataset_order,
    paper_column_order,
]
baseline_spearman_table.columns = paper_column_labels
baseline_spearman_table.loc["Average"] = (
    baseline_spearman_table.mean(axis=0)
)
baseline_spearman_table.index.name = "Dataset"

baseline_spearman_table_path = (
    RIYADI_UNCERTAINTY_OUTPUT_DIR
    / "table19_style_spearman_signed.csv"
)
baseline_spearman_table.to_csv(
    baseline_spearman_table_path
)

print("Baseline signed Spearman correlations:")
display(baseline_spearman_table.round(3))
print("Saved:", baseline_spearman_path)
print("Saved:", baseline_spearman_table_path)


Baseline signed Spearman correlations:


F1 Micro                          \
          Cosine Similarity Mean Fisher-Pearson Skewness   
Dataset                                                    
Financial                  0.036                   0.000   
EnvData                   -0.143                   0.286   
ACGNews                   -0.571                   0.464   
ECtHR                     -0.679                   0.357   
Ohsumed                   -0.714                   0.321   
Rheology                  -0.143                  -0.179   
arXiv                     -0.607                   0.536   
Average                   -0.403                   0.255   

                        F1 Macro                          
          Cosine Similarity Mean Fisher-Pearson Skewness  
Dataset                                                   
Financial                 -0.143                   0.071  
EnvData                   -0.143                   0.286  
ACGNews                   -0.571                   0.464  
ECtHR                     -0.250                   0.750  
Ohsumed                   -0.643                   0.464  
Rheology                  -0.143                  -0.179  
arXiv                     -0.679                   0.679  
Average                   -0.367                   0.362

Saved: /home/amyliu/Desktop/GR/Standardizing/results/riyadi_baseline_uncertainty10_reproduced/spearman_correlations.csv
Saved: /home/amyliu/Desktop/GR/Standardizing/results/riyadi_baseline_uncertainty10_reproduced/table19_style_spearman_signed.csv
